# L5: Data Analyst Agent 🤓

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> , notebooks and other files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

First start by filtering warnings and loading environment variables.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
cat helper.py

# Add your utilities or helper functions to this file.

import os
from dotenv import load_dotenv, find_dotenv

# these expect to find a .env file at the directory above the lesson.                                                                                                                     # the format for that file is (without the comment)                                                                                                                                       #API_KEYNAME=AStringThatIsTheLongAPIKeyFromSomeService                                                                                                                                     
def load_env():
    _ = load_dotenv(find_dotenv())

def get_openai_api_key():
    load_env()
    openai_api_key = os.getenv("OPENAI_API_KEY")
    return openai_api_key


In [ ]:
from helper import load_env
load_env()

Now, let's create the data analyst agent. You will create a sandbox, then open the `pokemon.csv` dataset and write it to the sandbox as `data.csv`.

In [3]:
from lib.utils import create_sandbox

sbx = create_sandbox()

with open("pokemon.csv", "r") as f:
    content = f.read()

sbx.files.write("data.csv", content)

INFO     [sandbox] 🚀 Creating new Sandbox.create(id=im70bm2j88qjkhnypc6ap)

WriteInfo(name='data.csv', type='file', path='/home/user/data.csv')

Now you can import your `coding_agent` and modify the system prompt to let it know that the user has uploaded a dataset called `data.csv` and it should create interesting plots.

In [4]:
cat lib/tools.py

import json
from typing import Callable, Optional
from e2b_code_interpreter import Execution, Sandbox


def execute_code(sbx: Sandbox, code: str, language: str = "python") -> Execution:
    execution = sbx.run_code(code, language)
    metadata = {}
    results = execution.results
    for result in results:
        if result.png:
            metadata["images"] = [result.png]
            result.png = None
            result.chart = None
    return execution.to_json(), metadata


tools = {
    "execute_code": execute_code,
    "execute_bash": lambda **a: execute_code(**a, language="bash"),
    "list_directory": lambda **a: execute_code(
        a["sbx"],
        f"list_directory(secure_path({repr(a.get('path', '.'))}), {repr(a.get('ignore'))}, {a.get('offset', 0)}, {a.get('limit', 16)})",
    ),
    "read_file": lambda **a: execute_code(
        a["sbx"],
        f"read_file(secure_path({repr(a.get('file_path', ''))}), {a.get('limit')}, {a.get('offset', 0)})",
    ),
    "write_file": lam

In [5]:
from lib.coding_agent import coding_agent, log
from lib.tools_schemas import execute_code_schema
from lib.tools import execute_code
from openai import OpenAI

client = OpenAI()

system = """You are a senior python programmer. 
You must run the code using the `execute_code` tool.
The user has uploaded a data.csv.
You help the user understanding the data 
by creating interesting plots.
"""

tools = { "execute_code" : execute_code }

Now you can query the data as normal and the sandbox will execute code on the data to answer the queries.

In [6]:
messages = []

query = "What is the data about?"

messages, usage = log(coding_agent,
    messages=messages,
    query=query,
    client=client,
    system=system,
    tools_schemas=[execute_code_schema],
    tools=tools,
    max_steps=10,
    sbx=sbx,
)

╭───────────────────────────────────────────────── 🤖 Tool Call ──────────────────────────────────────────────────╮
│ execute_code                                                                                                    │
│ Arguments: {"code":"import pandas as pd\n\ndf = pd.read_csv('data.csv')\ndf.head(), df.columns, df.info()"}     │
│ Result: {"results": [{"text": "(                     abilities  against_bug  against_dark  against_dragon  \\\n │
│ 0  ['Overgrow', 'Chlorophyll']          1.0           1.0             1.0   \n 1  ['Overgrow', '...             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFO     ✨: The data appears to be about Pokémon. It includes columns related to various Pokémon attributes such  
         as:                                                                                                       
                                                                                                                   
         - abilities                                                                                               
         - type (type1 and type2)                                                                                  
         - various "against" columns representing effectiveness against different types (e.g., against_bug,        
         against_fire, against_water, etc.)                                                                        
         - stats like attack, defense, sp_attack (special attack), sp_defense, speed, and hp                       
         - details such as height, weight, base happiness, capture rate, and experience growth                     
         - classification and name (including Japanese name)                                                       
         - generation and whether the Pokémon is legendary                                                         
                                                                                                                   
         Would you like me to create some visualizations to explore this data further?

INFO     [agent] 🔢 tokens: 2002 total

In [7]:
query = "Can you aggregate the pokemons by type?"

messages, usage = log(coding_agent,
    messages=messages,
    query=query,
    client=client,
    system=system,
    tools_schemas=[execute_code_schema],
    tools=tools,
    max_steps=10,
    sbx=sbx,
)

╭───────────────────────────────────────────────── 🤖 Tool Call ──────────────────────────────────────────────────╮
│ execute_code                                                                                                    │
│ Arguments: {"code":"type1_counts = df['type1'].value_counts()\ntype2_counts =                                   │
│ df['type2'].value_counts(dropna=True)\n\n# Combine counts of type1 and type2 to get total counts for each       │
│ type\nall_types_counts = ty...                                                                                  │
│ Result: {"results": [{"text": "water       131\nnormal      109\ngrass        98\nflying       98\npsychic      │
│ 82\nbug          77\npoison       66\nground       66\nfire         65\nrock         59\nfightin...             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFO     ✨: Here is the total count of Pokémon aggregated by their types (including both primary and secondary    
         types):                                                                                                   
                                                                                                                   
         - Water: 131                                                                                              
         - Normal: 109                                                                                             
         - Grass: 98                                                                                               
         - Flying: 98                                                                                              
         - Psychic: 82                                                                                             
         - Bug: 77                                                                                                 
         - Poison: 66                                                                                              
         - Ground: 66                                                                                              
         - Fire: 65                                                                                                
         - Rock: 59                                                                                                
         - Fighting: 53                                                                                            
         - Dark: 50                                                                                                
         - Electric: 48                                                                                            
         - Fairy: 47                                                                                               
         - Steel: 46                                                                                               
         - Dragon: 44                                                                                              
         - Ghost: 41                                                                                               
         - Ice: 38                                                                                                 
                                                                                                                   
         Would you like to see a plot of this data?

INFO     [agent] 🔢 tokens: 2405 total

### Gradio UI

You can use this provided Gradio UI to have a nicer user experience when chatting with your code generation agent. It will also give interesting information about the context stack, including the number of tokens used in the result, by the assistant, the user, and tools.

In [8]:
cat lib/ui.py

import gradio as gr
from gradio import ChatMessage
from itertools import chain
from PIL import Image
from io import BytesIO
import base64
import json
import tiktoken

from .coding_agent import clean_messages_for_llm
from gradio_browser import Browser
from gradio_aicontext import AIContext


def count_tokens(message: dict) -> int:
    encoding = tiktoken.encoding_for_model("gpt-4")
    return encoding.encode(json.dumps(clean_messages_for_llm([message]))).__len__()


def parse_openai_message(part) -> list[ChatMessage]:
    messages = []
    if "role" in part:
        if part["role"] == "user":
            messages.append(ChatMessage(role="user", content=part["content"]))
        elif part["role"] == "assistant" and part["content"]:
            if isinstance(part["content"], list):
                messages.append(
                    ChatMessage(role="assistant", content=part["content"][0]["text"])
                )
            else:
                messages.append(ChatMessage(role="assis

In [9]:
from lib.ui import ui

ui(coding_agent,
    messages,
    client=client,
    system=system,
    tools_schemas=[execute_code_schema],
    tools=tools,
    max_steps=10,
    sbx=sbx,
).launch(share=True, height=800)

* Running on local URL:  https://0.0.0.0:7860
* Running on public URL: https://78e3dde5c251bb6ae4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Your Turn

Now it's your turn to explore this unknown dataset. Ask questions and learn more about the information inside.

In [10]:
sbx = create_sandbox()
with open("unknown.csv", "rb") as f:
    content = f.read()
sbx.files.write("data.csv", content)

messages = []

ui(
    coding_agent,
    messages,
    client=client,
    system=system,
    tools_schemas=[execute_code_schema],
    tools=tools,
    max_steps=10,
    sbx=sbx,
).launch(share=True, height=800)

INFO     [sandbox] 🔌 Reconnecting to Sandbox.create(id=im70bm2j88qjkhnypc6ap)

* Running on local URL:  https://0.0.0.0:7861
* Running on public URL: https://e41786e8213e3b2ab6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
